# 🎯 Полётные задания

Пишем и тестируем миссии для дрона. MAVLink идёт из SITL прямо на Windows (UDP:14550) — WSL-терминалы не нужны.

**Перед началом**: в notebook `drone_control_ardupilot.ipynb` выполнить «МОСТ» (`bridge_up`) и «SITL» (`launch_sitl`). Firewall UDP 14550 должен быть разрешён (команда в том notebook).

**Порядок здесь**: ячейка «Подключение» → «Команды» → пиши задания.

**Основные команды**:
- `arm_and_takeoff(5)` — ARM + взлёт на 5м (ждёт набора)
- `goto(north, east, alt)` — лететь в точку (метры от старта, alt вверх)
- `land()` — посадка + disarm
- `where()` — где дрон сейчас
- `monitor(10)` — слушать телеметрию 10 сек

## 1. Подключение (ждёт heartbeat от SITL)

In [ ]:
from pymavlink import mavutil
import time, threading, math

conn = mavutil.mavlink_connection('udpin:0.0.0.0:14550', source_system=255)
print("Слушаем UDP 0.0.0.0:14550, ждём heartbeat...")

def _hb_loop():
    while True:
        try:
            conn.mav.heartbeat_send(
                mavutil.mavlink.MAV_TYPE_GCS,
                mavutil.mavlink.MAV_AUTOPILOT_INVALID, 0, 0, 0)
        except Exception:
            pass
        time.sleep(1)

threading.Thread(target=_hb_loop, daemon=True).start()

msg = conn.recv_match(type='HEARTBEAT', blocking=True, timeout=60)
if not msg:
    raise SystemExit("Нет heartbeat за 60 сек — SITL запущен? Firewall 14550 разрешён?")
print(f"ArduPilot онлайн! sysid={conn.target_system}")

# Просим нужные потоки телеметрии
for msg_id, interval_us in [(33, 200000), (32, 200000), (36, 200000)]:
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_SET_MESSAGE_INTERVAL, 0,
        msg_id, interval_us, 0, 0, 0, 0, 0)
print("Потоки телеметрии запрошены (GLOBAL_POSITION_INT, LOCAL_POSITION_NED, SERVO)")

## 2. Команды (выполнить один раз — дальше просто вызывать)

In [ ]:
GUIDED, LAND_MODE = 4, 9

def _drain_status(t=0.0):
    """Печатает накопившиеся STATUSTEXT (t сек)."""
    t0 = time.time()
    while time.time() - t0 <= t:
        m = conn.recv_match(type='STATUSTEXT', blocking=False)
        if m:
            print('  [AP]', m.text)
        else:
            time.sleep(0.05)

def set_mode(mode_num, name=""):
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_DO_SET_MODE, 0,
        mavutil.mavlink.MAV_MODE_FLAG_CUSTOM_MODE_ENABLED, mode_num, 0, 0, 0, 0, 0)
    time.sleep(1)
    print(f"Режим -> {name or mode_num}")

def wait_ekf(timeout=60):
    """Ждём готовность EKF: как только пошёл LOCAL_POSITION_NED — позиция есть.

    Без этого ARM падает с 'Need Position Estimate' / 'waiting for home'
    (EKF ставит origin через ~10-20 сек после старта SITL).
    """
    print("Ждём готовность EKF (origin + позиция)...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(blocking=True, timeout=1)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'LOCAL_POSITION_NED':
            print("EKF готов: позиция публикуется")
            time.sleep(2)  # небольшая стабилизация
            return True
    print("Timeout ожидания EKF — пробуем ARM всё равно")
    return False

def arm(attempts=3):
    """ARM с fallback на FORCE, несколько циклов (arming-чек 'Main loop slow' и т.п.)."""
    for cycle in range(attempts):
        for force in (False, True):
            label = 'FORCE ARM' if force else 'ARM'
            conn.mav.command_long_send(conn.target_system, 1,
                mavutil.mavlink.MAV_CMD_COMPONENT_ARM_DISARM, 0,
                1.0, 21196.0 if force else 0.0, 0, 0, 0, 0, 0)
            t0 = time.time()
            while time.time() - t0 < 5:
                m = conn.recv_match(blocking=True, timeout=1)
                if not m:
                    continue
                t = m.get_type()
                if t == 'STATUSTEXT':
                    print('  [AP]', m.text)
                elif t == 'HEARTBEAT' and (m.base_mode & mavutil.mavlink.MAV_MODE_FLAG_SAFETY_ARMED):
                    print(f"{label}: OK")
                    return True
                elif t == 'COMMAND_ACK' and m.command == mavutil.mavlink.MAV_CMD_COMPONENT_ARM_DISARM:
                    if m.result == 0:
                        print(f"{label}: OK (ACK)")
                        return True
                    break
        if cycle < attempts - 1:
            print(f"Попытка {cycle+1} не прошла, ждём 4 сек и повторяем...")
            time.sleep(4)
    print("ARM не прошёл")
    return False

def arm_and_takeoff(alt=5.0, timeout=60):
    """Ожидание EKF + ARM + NAV_TAKEOFF, ждёт набора высоты."""
    set_mode(GUIDED, 'GUIDED')
    wait_ekf()
    if not arm():
        return False
    set_mode(GUIDED, 'GUIDED')  # после FORCE ARM режим мог сброситься
    time.sleep(2)               # auto_armed
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_NAV_TAKEOFF, 0, 0, 0, 0, 0, 0, 0, alt)
    print(f"Взлёт на {alt}м...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(blocking=True, timeout=1)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'GLOBAL_POSITION_INT':
            a = m.relative_alt / 1000.0
            if a >= alt - 0.5:
                print(f"Высота {a:.2f}м — взлёт завершён")
                return True
    print("Timeout взлёта")
    return False

def where(quiet=False):
    """Позиция дрона: (north, east, alt_вверх) метров от старта."""
    m = conn.recv_match(type='LOCAL_POSITION_NED', blocking=True, timeout=3)
    if not m:
        print("Нет LOCAL_POSITION_NED (EKF ещё без origin?)")
        return None
    n, e, alt = m.x, m.y, -m.z
    if not quiet:
        print(f"Дрон: north={n:.2f}м east={e:.2f}м alt={alt:.2f}м")
    return (n, e, alt)

def goto(north, east, alt, tolerance=0.7, timeout=60):
    """Лететь в точку (метры от точки старта; alt — высота ВВЕРХ)."""
    conn.mav.set_position_target_local_ned_send(
        0, conn.target_system, 1,
        mavutil.mavlink.MAV_FRAME_LOCAL_NED,
        0b110111111000,          # только позиция
        north, east, -alt,
        0, 0, 0, 0, 0, 0, 0, 0)
    print(f"Лечу в (north={north}, east={east}, alt={alt})...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        p = where(quiet=True)
        if p is None:
            continue
        d = math.sqrt((p[0]-north)**2 + (p[1]-east)**2 + (p[2]-alt)**2)
        if d < tolerance:
            print(f"Прибыл: north={p[0]:.2f} east={p[1]:.2f} alt={p[2]:.2f} (до цели {d:.2f}м)")
            return True
        _drain_status(0)
        time.sleep(0.3)
    print("Timeout goto — дрон в", where(quiet=True))
    return False

def land(timeout=60):
    """Посадка (режим LAND), ждёт disarm."""
    set_mode(LAND_MODE, 'LAND')
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(type='HEARTBEAT', blocking=True, timeout=2)
        if m and not (m.base_mode & mavutil.mavlink.MAV_MODE_FLAG_SAFETY_ARMED):
            print("Сел и disarmed")
            return True
        _drain_status(0)
    print("Timeout посадки")
    return False

def monitor(seconds=10):
    """Слушать телеметрию N секунд (STATUSTEXT + высота)."""
    t0 = time.time()
    last_alt = None
    while time.time() - t0 < seconds:
        m = conn.recv_match(blocking=True, timeout=0.5)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'GLOBAL_POSITION_INT':
            a = m.relative_alt / 1000.0
            if last_alt is None or abs(a - last_alt) > 0.2:
                print(f"  [ALT] {a:.2f}м")
                last_alt = a

print("Команды готовы: arm_and_takeoff, goto, land, where, monitor")

## Задание 1: взлёт на 1.7м и зависание (внутри склада — потолок и стеллажи!)

In [ ]:
arm_and_takeoff(1.7)
where()

## Задание 2: квадрат 2×2м на высоте 1.7м и посадка

(запускать после Задания 1, пока дрон висит; следи по viewport, чтобы маршрут не шёл в стеллаж — если идёт, поменяй знаки/оси под свой склад)

In [ ]:
goto(2, 0, 1.7)
goto(2, 2, 1.7)
goto(0, 2, 1.7)
goto(0, 0, 1.7)
land()

## Песочница — пиши свои задания здесь

In [ ]:
# Пример:
# arm_and_takeoff(3)
# goto(5, 0, 3)
# land()